# Tower Component Detection - YOLOv8 Training
**Classes:** `monopole_tower`, `supporting_tower`

## Instructions:
1. Upload `dataset_merged.zip` to this Colab session (Files panel on left)
2. Run all cells in order
3. Download `best_tower_model.pt` when done

In [ ]:
# Cell 1: Install dependencies
!pip install ultralytics>=8.2.0 -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: Mount Google Drive (optional - for saving results)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Upload and extract dataset
# Option A: Upload zip file directly
import os
from google.colab import files

if not os.path.exists('/content/dataset_merged'):
    print('Upload dataset_merged.zip ...')
    uploaded = files.upload()
    !unzip -q dataset_merged.zip -d /content/
    print('Dataset extracted!')
else:
    print('Dataset already exists!')

# Fix data.yaml paths for Colab
yaml_content = '''path: /content/dataset_merged
train: images/train
val: images/val
test: images/test

nc: 2
names: ['monopole_tower', 'supporting_tower']
'''
with open('/content/dataset_merged/data.yaml', 'w') as f:
    f.write(yaml_content)
print('data.yaml updated for Colab paths')

# Verify
for split in ['train', 'val', 'test']:
    n = len(os.listdir(f'/content/dataset_merged/images/{split}'))
    print(f'  {split}: {n} images')

In [ ]:
# Cell 4: Train YOLOv8 (100 epochs on Colab GPU)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # YOLOv8-nano pretrained

results = model.train(
    data='/content/dataset_merged/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,           # T4 GPU handles 16 well
    patience=20,        # Early stop if no improvement for 20 epochs
    project='/content/runs',
    name='tower_final',
    device=0,
    exist_ok=True,
    
    # Optimizer
    optimizer='AdamW',
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.8,
    
    # Heavy augmentation for small dataset
    augment=True,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    flipud=0.5,
    fliplr=0.5,
    degrees=20.0,
    translate=0.2,
    scale=0.5,
    shear=5.0,
    perspective=0.001,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    erasing=0.3,
    
    plots=True,
    save=True,
    save_period=10,
    val=True,
    verbose=True,
    workers=2,
    seed=42,
)

print('Training complete!')

In [ ]:
# Cell 5: Evaluate the best model
from ultralytics import YOLO
import json

model = YOLO('/content/runs/tower_final/weights/best.pt')
metrics = model.val(data='/content/dataset_merged/data.yaml', split='val', plots=True)

print(f'\nmAP@50:     {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'mAP@50-95:  {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'Precision:  {metrics.box.mp:.4f}  ({metrics.box.mp*100:.1f}%)')
print(f'Recall:     {metrics.box.mr:.4f}  ({metrics.box.mr*100:.1f}%)')

# Per-class
for i, name in model.names.items():
    if i < len(metrics.box.p):
        print(f'  [{name}] P={metrics.box.p[i]:.3f} R={metrics.box.r[i]:.3f} mAP50={metrics.box.ap50[i]:.3f}')

# Save metrics
summary = {
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50_95': round(float(metrics.box.map), 4),
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
}
with open('/content/runs/tower_final/evaluation_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nMetrics saved!')

In [ ]:
# Cell 6: Display training graphs
from IPython.display import Image, display
import os

run_dir = '/content/runs/tower_final'
plots = ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'BoxF1_curve.png']

for p in plots:
    path = os.path.join(run_dir, p)
    if os.path.exists(path):
        print(f'\n--- {p} ---')
        display(Image(filename=path, width=800))

In [ ]:
# Cell 7: Download the trained model
import shutil
from google.colab import files

# Copy to convenient location
shutil.copy('/content/runs/tower_final/weights/best.pt', '/content/best_tower_model.pt')

# Also copy to Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs('/content/drive/MyDrive/tower_detection', exist_ok=True)
    shutil.copy('/content/best_tower_model.pt', '/content/drive/MyDrive/tower_detection/best_tower_model.pt')
    print('Model saved to Google Drive: tower_detection/best_tower_model.pt')

# Download directly
files.download('/content/best_tower_model.pt')
print('Download started!')